In [1]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import mlflow
import numpy as np
import pandas as pd
from prefect import task, flow
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("RandomState-hyperopt")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1773843100059, experiment_id='1', last_update_time=1773843100059, lifecycle_stage='active', name='RandomState-hyperopt', tags={}, workspace='default'>

In [3]:
file_path = './data/yellow_tripdata_2023-03.parquet'
def load(file: str) -> pd.DataFrame:
    df = pd.read_parquet(file).head(500)
    df['duration'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    # df.head(5)
    return df

In [4]:
df = load(file_path)

In [5]:
def preprocess(df: pd.DataFrame, fit_dv: bool=True) -> tuple:
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    df_encoded = pd.get_dummies(df[['PU_DO', 'trip_distance']], columns=['PU_DO'])
    return df_encoded

In [6]:
y_train = df['duration'].values

In [7]:
X = preprocess(df)

In [8]:
X_train, X_val, Y_train, Y_val = train_test_split(X, y_train, test_size=0.2, random_state=42)

In [17]:
search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

def train(params: dict) -> dict:
    with mlflow.start_run():
        mlflow.set_tag("author", "ahmed")
        mlflow.sklearn.autolog()
        
        model = RandomForestRegressor(**params)
        model.fit(X_train, Y_train)
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_pred, Y_val)

        mlflow.log_metrics({"rmse": rmse})

    return {"loss": rmse, "status": STATUS_OK}

rstate = np.random.default_rng(42)

fmin(
    fn=train,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials(),
    rstate=rstate
)

  0%|                                                                                                                                    | 0/10 [00:00<?, ?trial/s, best loss=?]

2026/03/18 19:09:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run valuable-carp-328 at: http://127.0.0.1:5000/#/experiments/1/runs/5b75423101164ccfacb4e5e747c7bf56                                                                   

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                                                                                                                    

 10%|██████████▋                                                                                                | 1/10 [00:04<00:36,  4.04s/trial, best loss: 3.782162447681632]

2026/03/18 19:09:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run brawny-ram-925 at: http://127.0.0.1:5000/#/experiments/1/runs/65beb7111f5c4fb5b7becd1b3bb9c075                                                                      

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                                                                                                                    

 20%|█████████████████████▏                                                                                    | 2/10 [00:07<00:30,  3.83s/trial, best loss: 3.7536664272535853]

...

2026/03/18 19:09:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run melodic-eel-61 at: http://127.0.0.1:5000/#/experiments/1/runs/82931311b68e4d30bd5cfd91128ab194                                                                      

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                                                                                                                    

 90%|█████████████████████████████████████████████████████████████████████████████████████████████████▏          | 9/10 [00:34<00:03,  3.89s/trial, best loss: 3.50128275468236]

2026/03/18 19:09:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run hilarious-flea-219 at: http://127.0.0.1:5000/#/experiments/1/runs/2bbf601a76cb4ddfa1700111da1a08f9                                                                  

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                                                                                                                    

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:38<00:00,  3.83s/trial, best loss: 3.50128275468236]


{'max_depth': np.float64(18.0),
 'min_samples_leaf': np.float64(1.0),
 'min_samples_split': np.float64(10.0),
 'n_estimators': np.float64(38.0)}